# Question 11 - 274. H-Index

Given an array of integers `citations` where `citations[i]` is the number of citations a researcher received for their `i-th` paper, return the researcher's **h-index**.

According to the definition of h-index on Wikipedia: The h-index is defined as the maximum value of `h` such that the given researcher has published at least `h` papers that have each been cited at least `h` times.

**Example 1:**

    Input: citations = [3,0,6,1,5]
    Output: 3
    Explanation: [3,0,6,1,5] means the researcher has 5 papers in total and each of them had received 3, 0, 6, 1, 5 citations respectively.
    Since the researcher has 3 papers with at least 3 citations each and the remaining two with no more than 3 citations each, their h-index is 3.

**Example 2:**

    Input: citations = [1,3,1]
    Output: 1

**Constraints:**

- `n == citations.length`
- `1 <= n <= 5000`
- `0 <= citations[i] <= 1000`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Counting sort / bucket by value (capped at n)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(n²) | O(1) |
| 1½ Sort | O(n log n) | O(1)–O(n) |
| 2️⃣ Optimized | O(n) | O(n) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Restating: I want the largest `h` such that **at least `h` papers have ≥ `h` citations each**."*

- Range of `h`? → `0 ≤ h ≤ n`. You can't have more than `n` papers. **This upper bound is the key to the optimal solution.**
- All zeros? → `h = 0`.
- Can I sort / modify the input? → I'll ask. If not, I'll copy it or use a separate count array.
- Is the input already sorted? → No. (If it were, that's LC 275 H-Index II, solved with binary search.)

### Step 1 · Brute force

**Idea:** Try every candidate `h` from `n` down to `0`. For each, count how many papers have `≥ h` citations. The first `h` that works is the answer.

🗣️ *"The definition maps directly to code: for each candidate h, count papers with at least h citations. n+1 candidates times an O(n) count gives O(n²). With n = 5000 that's 25 million operations, which works but isn't elegant."*

In [1]:
class SolutionBrute:
    def hIndex(self, citations: list[int]) -> int:
        n = len(citations)
        for h in range(n, -1, -1):
            if sum(1 for c in citations if c >= h) >= h:
                return h
        return 0

#### Step 1½ · Sort descending

After sorting descending, the paper at position `i` (0-indexed) is the `(i+1)`-th most cited. If `citations[i] ≥ i + 1`, then the top `i+1` papers all have ≥ `i+1` citations, so `h ≥ i+1`. Walk until that breaks.

🗣️ *"Sorting puts the papers in the order the definition cares about, so the answer becomes a single scan. That's O(n log n). Can I avoid the comparison sort?"*

In [2]:
class SolutionSort:
    def hIndex(self, citations: list[int]) -> int:
        cs = sorted(citations, reverse=True)
        h = 0
        while h < len(cs) and cs[h] >= h + 1:
            h += 1
        return h

### Step 2 · Optimize

**Bottleneck:** Comparison sort costs O(n log n). But we don't need a full ordering. We only need to know, for each `h`, **how many papers have ≥ h citations**.

**Key insight:** `h` can never exceed `n`. So a paper with 1000 citations is no different from one with `n` citations, because both count toward every possible `h`. **Cap every value at `n`**, then count with an array of size `n + 1` (a counting sort).

Then walk `h` from `n` down to `0` with a running total `papers = count[h] + count[h+1] + … + count[n]` (papers with ≥ h citations). The first time `papers ≥ h`, that `h` is the answer.

🗣️ *"Because h is bounded by n, I can bucket citations into n+1 buckets, capping big values at n. Then I sweep from the top, accumulating how many papers have at least h citations. The first h where that count reaches h is the answer. That's O(n) time and O(n) space."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Check every `h` (brute) | ❌ | O(n²). It recounts the whole array for every candidate. |
| Comparison sort + scan | ✅ (good) | O(n log n), short code, and O(1) extra if sorting in place is allowed. It's the safest answer to say first. |
| Binary search on `h` | ⚠️ | "≥ h papers with ≥ h" is monotone in h, so it works, but each check is O(n), giving O(n log n) with no gain over sorting. |
| Hash map of counts | ⚠️ | Works, but values are small integers in a known range, so an array is simpler and faster. |
| **Count array of size n+1 (cap at n)** | ✅ (best) | Uses the bound `h ≤ n` to get linear time, without needing a general sort. |
| Heap | ❌ | No benefit. We'd pop up to n items at O(log n) each. |

**Trade-off to say out loud:** the count array uses **O(n) extra space** to buy O(n) time. If memory is tight or input can be sorted in place, the sort version is O(1) extra space.

In [3]:
class Solution:
    def hIndex(self, citations: list[int]) -> int:
        n = len(citations)
        count = [0] * (n + 1)
        for c in citations:
            count[min(c, n)] += 1     # cap: anything >= n counts for every h

        papers = 0                    # papers with >= h citations
        for h in range(n, -1, -1):
            papers += count[h]
            if papers >= h:
                return h
        return 0

### Step 3 · Toy example walkthrough

`citations = [3, 0, 6, 1, 5]`, n = 5

**Bucket (cap at 5):** 3→3, 0→0, 6→**5** (capped), 1→1, 5→5

| index h | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| count[h] | 1 | 1 | 0 | 1 | 0 | 2 |

**Sweep from the top:**

| h | count[h] | papers (≥ h) | papers ≥ h? |
|---|---|---|---|
| 5 | 2 | 2 | 2 ≥ 5 ❌ |
| 4 | 0 | 2 | 2 ≥ 4 ❌ |
| 3 | 1 | **3** | 3 ≥ 3 ✅ → return **3** |

Sanity check: papers with ≥ 3 citations are {3, 6, 5}, which is 3 papers ✅. And there aren't 4 papers with ≥ 4 ✅.

In [4]:
cases = [
    ([3, 0, 6, 1, 5], 3),
    ([1, 3, 1], 1),
    ([0], 0),
    ([0, 0, 0], 0),
    ([100], 1),
    ([1000, 1000, 1000], 3),
    ([4, 4, 0, 0], 2),
    ([11, 15], 2),
]
for cits, expected in cases:
    assert SolutionBrute().hIndex(cits) == expected
    assert SolutionSort().hIndex(cits) == expected
    assert Solution().hIndex(cits) == expected
    print(cits, "->", expected)

import random
for _ in range(500):
    cits = [random.randint(0, 12) for _ in range(random.randint(1, 10))]
    assert Solution().hIndex(cits) == SolutionBrute().hIndex(cits), cits
print("All tests passed ✅")

[3, 0, 6, 1, 5] -> 3
[1, 3, 1] -> 1
[0] -> 0
[0, 0, 0] -> 0
[100] -> 1
[1000, 1000, 1000] -> 3
[4, 4, 0, 0] -> 2
[11, 15] -> 2
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Brute force | O(n²) | O(1) | n+1 candidates × O(n) count |
| Sort + scan | O(n log n) | O(1) in place / O(n) copy | Sorting dominates |
| **Counting (cap at n)** | **O(n)** | **O(n)** | One pass to bucket and one pass to sweep |

**Complexity, explained:** bucketing loops over the `n` citations once. The sweep loops over at most `n+1` buckets. Each step is O(1), so the total is O(n). The count array has `n+1` slots, so space is O(n).

**Edge cases:** all zeros → 0. One paper with many citations → 1 (capping matters here). Huge values everywhere → `n`.

**Follow-ups:**
- *Input already sorted ascending* (LC 275) → binary search for the first `i` with `citations[i] ≥ n − i`. O(log n).
- *Streaming citations?* → Keep the count array and the running `h`. When a paper is added, `h` can only grow, by at most 1.

🗣️ **30-second recap:** *"Brute force checks every h in O(n²). Sorting descending makes it one scan, O(n log n). The trick for O(n) is that h is at most n, so I cap citations at n and count them in n+1 buckets, then sweep from high to low, accumulating papers with at least h citations. The first h where the count reaches h is the answer."*

---
## 📎 Appendix: original reference solution

*Kept from the source repo for comparison.*

In [5]:
class Solution:
    # O(n logn) time complexity: sorting
    # O(1) space complexity: in-place sorting
    def hIndex(self, citations: list[int]) -> int:
        # sort the citations in descending order
        citations.sort(reverse=True)

        # initialize the h-index to 0
        h_index = 0

        # iterate through the citations
        for i in range(len(citations)):
            # if the citation is greater than or equal to the h-index
            if citations[i] >= h_index + 1:
                # increment the h-index
                h_index += 1
            # otherwise
            else:
                # break out of the loop to avoid unnecessary iterations
                break

        # return the h-index
        return h_index

In [6]:
# Test cases

h_index = Solution().hIndex

test_case_1 = [3, 0, 6, 1, 5]
test_case_2 = [1, 3, 1]

output_1 = h_index(test_case_1)
output_2 = h_index(test_case_2)

output_1, output_2

(3, 1)